# QTCAD: Semiconductor Quantum Device Simulation

## What is QTCAD?

**QTCAD** (Quantum Technology Computer-Aided Design) is a commercial simulation platform developed by Nanoacademic Technologies, specialized in modeling semiconductor quantum devices, particularly spin qubits in quantum dots. It is the first commercial finite-element software designed specifically for spin qubits, enabling virtual prototyping and device behavior prediction before fabrication.

## Core Physical Capabilities

QTCAD integrates multiple physics solvers that work hierarchically:

1. **Electrostatics (Nonlinear Poisson):** Calculates the electrostatic potential throughout the device considering materials, doping, and applied voltages. Its adaptive meshing technology ensures convergence even at cryogenic temperatures (millikelvin).

2. **Quantum Confinement (Schrödinger):** Solves the Schrödinger equation within the effective mass approximation to obtain energy levels and wavefunctions of confined electrons.

3. **Many-Body Physics (Configuration Interaction):** Explicitly treats electron-electron interactions through exact diagonalization, calculating energy spectra and addition energies.

## Key Applications for This Project

The most relevant capability is **charge stability diagram generation**. Combining the above solvers with a quantum transport model, QTCAD predicts electron occupancy in double quantum dots as gate voltages are swept, producing diagrams equivalent to experimental measurements.

These diagrams show charge regions (e.g., (0,0), (1,0), (0,1), (1,1)) and constitute the device's "fingerprint." Recent research has used QTCAD to model two-qubit gates in silicon devices, including charge noise effects.

## Role in This Project

QTCAD serves as the **data generation engine**. Through its Python API, we will automate voltage sweeps to produce a labeled dataset of stability diagrams. These physically-grounded images will feed the machine learning pipeline: first as a classical baseline (SVM with RBF kernel), then as a simulated quantum kernel (ZZFeatureMap) using Qiskit.

Thus, QTCAD bridges quantum device physics simulation with machine learning analysis, closing the loop between quantum hardware and algorithms.

## Physical Foundations: From Electrical Control to Quantum Dots in FD-SOI

To understand the simulations we perform with QTCAD, it's helpful to build a step-by-step picture: from the classical control of a device, through the quantum mechanics of confinement, to the specific technology we use.

### 1. The Language of Control: Voltages and Gates

Every electronic device, classical or quantum, is controlled by applying voltages. In quantum dot devices, these voltages are applied to small metallic terminals called **gates**. Each type of gate has a specific function to "sculpt" the electrical landscape inside the device:

- **Barrier Gates:** Act as "valves" that control the height of potential barriers. Increasing the voltage on these gates lowers the barrier, allowing electrons to tunnel from one region to another.
- **Plunger Gates:** Function as "keys" that adjust the energy of a specific quantum dot. Raising or lowering their voltage allows us to "fill" or "empty" the dot, one electron at a time.
- **Back Gate:** A global gate located beneath the device that helps create the electron layer in the silicon channel, providing background control over the entire system.

The combination of these voltages allows us to engineer a custom electrostatic potential, defining where and how electrons will be confined.

### 2. The Quantum Concept: Quantum Dots

When we confine electrons in a region so small (a few nanometers), their behavior ceases to be classical and begins to be governed by the laws of quantum mechanics. This confinement region is known as a **quantum dot** (or "artificial atom").

The key properties of a quantum dot are:

- **3D Confinement:** The electron is trapped in all three spatial dimensions.
- **Energy Quantization:** Just like in an atom, electrons can only occupy specific, discrete energy levels (E₁, E₂, E₃, ...).
- **Charge in Discrete Units:** The energy required to add one more electron to the dot (the charging energy, E_c = e²/2C) can be enormous at the nanoscale. This prevents electrons from freely entering or leaving (a phenomenon known as **Coulomb blockade**), allowing us to control the dot's charge, electron by electron. In a **double quantum dot** system, like the one we simulate, the charge state is described as (N, M), where N and M are the number of electrons in the first and second dot, respectively.

### 3. The Heart of the Simulation: Equations Solved by QTCAD

To predict the behavior of these quantum dots, QTCAD solves a set of physical equations hierarchically, connecting the classical world of voltages with the quantum world of electrons.

#### Nonlinear Poisson Equation (The Classical Potential)
The first step is to calculate the electrical potential, φ, throughout the device using the **Poisson equation**:

∇ · (ε ∇φ) = -ρ(φ)

This equation tells us how the potential distributes itself based on the materials (ε, the dielectric constant), fixed charges (ρ), and crucially, the voltages we apply to the gates. This equation is "nonlinear" because ρ itself depends on φ.

#### Schrödinger Equation (The Quantum States)
The calculated potential, φ, acts as the "stage" that electrons see. To find their possible energy levels and their wavefunctions (the shape of their probability "cloud"), we solve the **time-independent Schrödinger equation** within the quantum dot:

(-ħ²/2) ∇ · ( (1/m*) ∇ψ ) + V_conf(r) ψ_i(r) = E_i ψ_i(r)

Here:
- ħ is the reduced Planck constant
- m* is the effective mass of the electron in silicon (a simplification called the **effective mass approximation**)
- ψ_i is the wavefunction of the i-th state (|ψ_i|² gives the probability of finding the electron at a given position)
- V_conf is the confining potential derived from φ
- E_i is the energy of the i-th state

This equation gives us the discrete "energy levels" of the dot and their corresponding wavefunctions.

#### Many-Body Physics (Electron-Electron Interactions)
When we have more than one electron in the system, they are not independent: they repel each other. QTCAD treats this problem with a **Configuration Interaction** method (also known as exact diagonalization). Starting from the single-particle states obtained from the Schrödinger equation, it builds a Hamiltonian that includes both kinetic energy and the mutual Coulomb repulsion between electrons. Diagonalizing this Hamiltonian yields the total energies of the N-electron system. The difference between total energies for N and N+1 electrons gives us the electrochemical potential, allowing us to predict which charge state (e.g., (1,0) or (1,1)) the system will be in for each combination of voltages.

This hierarchical approach—from classical electrostatics to single-particle quantum mechanics to many-body physics—is the core of quantum device simulation with QTCAD, and it ultimately generates the **charge stability diagrams** that will be the foundation of our machine learning study.

### 4. The Technological Stage: FD-SOI (Fully Depleted Silicon on Insulator)

Our quantum dot does not float in a vacuum; it is built on a real and highly promising technological platform for quantum computing: **FD-SOI** technology.

- **Structure:** An FD-SOI device consists of an ultra-thin layer of silicon (the "channel") deposited on top of a **Buried Oxide (BOX)** layer, typically just a few nanometers thick. This oxide layer insulates the channel from the bulk silicon substrate below.
- **Quantum Advantages:** This structure allows for exceptional electrostatic control and minimizes current leakage. The ultra-thin silicon layer creates a strong vertical confinement, which is ideal for forming well-defined quantum dots. It is the same technology used in the high-performance microprocessor industry (CMOS), making it one of the most promising pathways for fabricating spin qubits in a scalable manner.
- **In Our Simulation:** The mesh we load (`dqdfdsoi.msh`) precisely models a double quantum dot in this FD-SOI architecture. It defines specific regions such as the silicon channel (`channel`), the buried oxide (`buried_oxide`), the gate oxides (`oxide`), and the various metallic gates (`source`, `drain`, and the barrier and plunger gates).

### 5. Workflow Summary

1.  **We apply voltages** to the gates (barrier gates, plunger gates, back gate).
2.  **QTCAD solves the Poisson equation** to obtain the classical electrostatic potential throughout the device.
3.  **It solves the Schrödinger equation** in the quantum dot regions to find the discrete energy levels and wavefunctions.
4.  **It applies the Configuration Interaction method** to calculate electron-electron interactions and determine the total energy for different charge configurations.
5.  By sweeping the plunger gate voltages and repeating this process, we construct **charge stability diagrams**: 2D maps that relate the applied voltages to the charge state (N, M) of the double quantum dot.

These diagrams constitute the labeled dataset that will feed our subsequent machine learning analysis, both classical (SVM with RBF kernel) and quantum (simulated quantum kernel with ZZFeatureMap).